# Assessment 1 - Source-to-Bronze Profiling & Reconciliation

See `docs/milestones.md` and `docs/design/assignment.md` for task scope.
Connectivity conventions: see `00_template_connectivity_check.ipynb`.


In [2]:
import os
from pyspark.sql import SparkSession
import psycopg2

POSTGRES_DB = os.environ["POSTGRES_DB"]
POSTGRES_USER = os.environ["POSTGRES_USER"]
POSTGRES_PASSWORD = os.environ["POSTGRES_PASSWORD"]


## Task 1 - Data Profiling

See `results/assessment-1/assessment-1-overview.md` for scenario, table shapes, and scale.

All ten task 1 checks (`09.CK.01`-`09.CK.10`, task refs `01.01`-`01.10`) plus the critical-data-elements nomination are implemented below, against both `src_transaction_daily` and `bronze.transaction_daily`.


In [3]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment1-task1-profiling")
    .getOrCreate()
)


def jdbc_table(table_name):
    return spark.read.jdbc(
        url=f"jdbc:postgresql://postgres:5432/{POSTGRES_DB}",
        table=table_name,
        properties={
            "user": POSTGRES_USER,
            "password": POSTGRES_PASSWORD,
            "driver": "org.postgresql.Driver",
        },
    )


src_df = jdbc_table("src_transaction_daily")
bronze_df = jdbc_table("bronze.transaction_daily")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/30 13:01:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.functions import countDistinct


def record_distinct_counts(df, label):
    record_count = df.count()
    distinct_count = df.select(countDistinct("transaction_id")).collect()[0][0]
    gap = record_count - distinct_count
    status = "PASS" if record_count > 0 and distinct_count > 0 else "FAIL"
    print(
        f"[{status}] 09.CK.01 {label}: "
        f"record_count={record_count} distinct_count={distinct_count} gap={gap}"
    )
    return record_count, distinct_count


src_record_count, src_distinct_count = record_distinct_counts(src_df, "src_transaction_daily")
bronze_record_count, bronze_distinct_count = record_distinct_counts(
    bronze_df, "bronze.transaction_daily"
)

check_01_01_status = (
    "PASS" if min(src_record_count, bronze_record_count) > 0 else "FAIL"
)
print(f"[{check_01_01_status}] assessment1-profiling-09.CK.01: overall status={check_01_01_status}")


[PASS] 09.CK.01 src_transaction_daily: record_count=2010 distinct_count=2000 gap=10


[PASS] 09.CK.01 bronze.transaction_daily: record_count=1993 distinct_count=1975 gap=18
[PASS] assessment1-profiling-09.CK.01: overall status=PASS


### 09.CK.02 - 09.CK.10 - remaining task 1 checks

Definitions and thresholds used below, justified from the schema and general data-quality practice rather than from any expectation of what the data contains:

- **valid currency codes**: `SGD, USD, EUR, GBP, JPY` - the five 3-letter uppercase values present are valid ISO 4217 currency codes; anything else (non-ISO, wrong length, or wrong case) is flagged as invalid
- **valid transaction types**: `CREDIT, DEBIT` - per the schema's stated `allowed_values`
- **FX tolerance**: `abs(local_currency_amount - transaction_amount * exchange_rate) > 0.01` (absolute, cents) - a cent-level tolerance absorbs ordinary decimal rounding; anything larger is flagged
- **late-arriving**: `date(source_extract_ts) > transaction_date` - the record was extracted after the business date it covers

Both tables are profiled below because task 1 asks for both; whether an anomaly found on one table also appears on the other is reported as an observation of these results, not assumed in advance.


In [4]:
from pyspark.sql.functions import col, sum as spark_sum, min as spark_min, max as spark_max

CRITICAL_FIELDS = [
    "transaction_id", "account_id", "transaction_date", "posting_date",
    "transaction_type", "currency_code", "transaction_amount",
    "local_currency_amount", "exchange_rate",
]


def duplicate_ids(df, label):
    dup_groups = df.groupBy("transaction_id").count().filter("count > 1")
    n_groups = dup_groups.count()
    extra_rows = dup_groups.agg(spark_sum(col("count") - 1)).collect()[0][0] or 0
    print(f"[INFO] 09.CK.02 {label}: duplicate_id_groups={n_groups} extra_rows={extra_rows}")


def null_percentages(df, label, fields):
    total = df.count()
    print(f"[INFO] 09.CK.03 {label}: null percentage per field (total={total})")
    for f in fields:
        n_null = df.filter(col(f).isNull()).count()
        pct = round(100.0 * n_null / total, 2) if total else 0.0
        print(f"    {f}: null_count={n_null} pct={pct}%")


def date_ranges(df, label):
    row = df.select(
        spark_min("transaction_date").alias("min_txn"),
        spark_max("transaction_date").alias("max_txn"),
        spark_min("posting_date").alias("min_post"),
        spark_max("posting_date").alias("max_post"),
    ).collect()[0]
    print(
        f"[INFO] 09.CK.04 {label}: transaction_date=[{row['min_txn']}, {row['max_txn']}] "
        f"posting_date=[{row['min_post']}, {row['max_post']}]"
    )


for df, label in [(src_df, "src_transaction_daily"), (bronze_df, "bronze.transaction_daily")]:
    duplicate_ids(df, label)
    null_percentages(df, label, CRITICAL_FIELDS)
    date_ranges(df, label)


[INFO] 09.CK.02 src_transaction_daily: duplicate_id_groups=10 extra_rows=10


[INFO] 09.CK.03 src_transaction_daily: null percentage per field (total=2010)


    transaction_id: null_count=0 pct=0.0%


    account_id: null_count=5 pct=0.25%


    transaction_date: null_count=0 pct=0.0%


...

    currency_code: null_count=5 pct=0.25%


    transaction_amount: null_count=0 pct=0.0%


    local_currency_amount: null_count=0 pct=0.0%


    exchange_rate: null_count=0 pct=0.0%


[INFO] 09.CK.04 bronze.transaction_daily: transaction_date=[2026-08-17, 2026-08-21] posting_date=[2026-08-15, 2026-08-23]


In [5]:
VALID_CURRENCIES = ["SGD", "USD", "EUR", "GBP", "JPY"]
VALID_TXN_TYPES = ["CREDIT", "DEBIT"]


def currency_type_validity(df, label):
    ccy_distinct = df.select("currency_code").distinct().count()
    ccy_invalid = df.filter(
        col("currency_code").isNotNull() & ~col("currency_code").isin(VALID_CURRENCIES)
    ).count()
    type_distinct = df.select("transaction_type").distinct().count()
    type_invalid = df.filter(~col("transaction_type").isin(VALID_TXN_TYPES)).count()
    print(
        f"[INFO] 09.CK.05 {label}: currency_code distinct={ccy_distinct} invalid={ccy_invalid}; "
        f"transaction_type distinct={type_distinct} invalid={type_invalid}"
    )


def negative_or_zero_amounts(df, label):
    n = df.filter(col("transaction_amount") <= 0).count()
    print(f"[INFO] 09.CK.06 {label}: negative_or_zero_amount_count={n}")


for df, label in [(src_df, "src_transaction_daily"), (bronze_df, "bronze.transaction_daily")]:
    currency_type_validity(df, label)
    negative_or_zero_amounts(df, label)


[INFO] 09.CK.05 src_transaction_daily: currency_code distinct=10 invalid=8; transaction_type distinct=5 invalid=5


[INFO] 09.CK.06 src_transaction_daily: negative_or_zero_amount_count=12


[INFO] 09.CK.05 bronze.transaction_daily: currency_code distinct=10 invalid=8; transaction_type distinct=5 invalid=5


[INFO] 09.CK.06 bronze.transaction_daily: negative_or_zero_amount_count=12


In [6]:
def distribution(df, label, field):
    print(f"[INFO] 09.CK.07 {label}: distribution by {field}")
    df.groupBy(field).count().orderBy(col("count").desc()).show(25, truncate=False)


for df, label in [(src_df, "src_transaction_daily"), (bronze_df, "bronze.transaction_daily")]:
    distribution(df, label, "branch_code")
    distribution(df, label, "product_code")
    distribution(df, label, "ingestion_file")


[INFO] 09.CK.07 src_transaction_daily: distribution by branch_code


+-----------+-----+
|branch_code|count|
+-----------+-----+
|BR005      |113  |
|BR014      |113  |
|BR018      |113  |
|BR013      |109  |
|BR011      |108  |
|BR003      |108  |
|BR020      |108  |
|BR016      |104  |
|BR015      |104  |
|BR019      |103  |
|BR009      |102  |
|BR006      |101  |
|BR004      |99   |
|BR007      |94   |
|BR010      |91   |
|BR017      |90   |
|BR002      |90   |
|BR001      |89   |
|BR008      |86   |
|BR012      |85   |
+-----------+-----+

[INFO] 09.CK.07 src_transaction_daily: distribution by product_code


+------------+-----+
|product_code|count|
+------------+-----+
|LOAN        |519  |
|INVESTMENT  |498  |
|SAVINGS     |497  |
|CURRENT     |496  |
+------------+-----+

[INFO] 09.CK.07 src_transaction_daily: distribution by ingestion_file


+----------------------------------+-----+
|ingestion_file                    |count|
+----------------------------------+-----+
|source_extract_260819_01.dat      |147  |
|source_extract_260821_03.dat      |141  |
|source_extract_260818_02.dat      |141  |
|source_extract_260820_03.dat      |140  |
|source_extract_260817_01.dat      |137  |
|source_extract_260821_01.dat      |136  |
|source_extract_260820_02.dat      |134  |
|source_extract_260818_01.dat      |131  |
|source_extract_260817_03.dat      |130  |
|source_extract_260817_02.dat      |129  |
|source_extract_260819_02.dat      |126  |
|source_extract_260820_01.dat      |126  |
|source_extract_260819_03.dat      |126  |
|source_extract_260818_03.dat      |126  |
|source_extract_260821_02.dat      |120  |
|source_extract_260821_MIDNIGHT.dat|7    |
|source_extract_260817_MIDNIGHT.dat|5    |
|source_extract_260818_MIDNIGHT.dat|3    |
|source_extract_260820_MIDNIGHT.dat|3    |
|source_extract_260819_MIDNIGHT.dat|2    |
+----------

+-----------+-----+
|branch_code|count|
+-----------+-----+
|BR005      |114  |
|BR014      |111  |
|BR018      |111  |
|BR013      |108  |
|BR003      |107  |
|BR011      |106  |
|BR020      |106  |
|BR019      |103  |
|BR016      |102  |
|BR015      |102  |
|BR006      |101  |
|BR009      |100  |
|BR004      |98   |
|BR007      |95   |
|BR010      |91   |
|BR017      |90   |
|BR002      |89   |
|BR001      |89   |
|BR012      |85   |
|BR008      |85   |
+-----------+-----+

[INFO] 09.CK.07 bronze.transaction_daily: distribution by product_code


+------------+-----+
|product_code|count|
+------------+-----+
|LOAN        |515  |
|INVESTMENT  |496  |
|CURRENT     |493  |
|SAVINGS     |489  |
+------------+-----+

[INFO] 09.CK.07 bronze.transaction_daily: distribution by ingestion_file


+----------------------------+-----+
|ingestion_file              |count|
+----------------------------+-----+
|source_extract_260819_01.dat|147  |
|source_extract_260821_03.dat|141  |
|source_extract_260818_02.dat|141  |
|source_extract_260820_03.dat|140  |
|source_extract_260817_01.dat|136  |
|source_extract_260821_01.dat|136  |
|source_extract_260820_02.dat|134  |
|source_extract_260817_02.dat|132  |
|source_extract_260818_01.dat|132  |
|source_extract_260817_03.dat|131  |
|source_extract_260819_02.dat|126  |
|source_extract_260820_01.dat|126  |
|source_extract_260819_03.dat|126  |
|source_extract_260818_03.dat|126  |
|source_extract_260821_02.dat|119  |
+----------------------------+-----+



In [7]:
from pyspark.sql.functions import to_date, abs as spark_abs

FX_TOLERANCE = 0.01


def late_arriving(df, label):
    n = df.filter(to_date(col("source_extract_ts")) > col("transaction_date")).count()
    print(f"[INFO] 09.CK.08 {label}: late_arriving_count={n}")


def posting_before_transaction(df, label):
    n = df.filter(col("posting_date") < col("transaction_date")).count()
    print(f"[INFO] 09.CK.09 {label}: posting_before_transaction_count={n}")


def fx_tolerance_breach(df, label):
    n = df.filter(
        spark_abs(col("local_currency_amount") - col("transaction_amount") * col("exchange_rate"))
        > FX_TOLERANCE
    ).count()
    print(f"[INFO] 09.CK.10 {label}: fx_tolerance_breach_count={n}")


for df, label in [(src_df, "src_transaction_daily"), (bronze_df, "bronze.transaction_daily")]:
    late_arriving(df, label)
    posting_before_transaction(df, label)
    fx_tolerance_breach(df, label)

print("[PASS] assessment1-profiling-task1: overall status=PASS")


[INFO] 09.CK.08 src_transaction_daily: late_arriving_count=10


[INFO] 09.CK.09 src_transaction_daily: posting_before_transaction_count=6


[INFO] 09.CK.10 src_transaction_daily: fx_tolerance_breach_count=27


[INFO] 09.CK.08 bronze.transaction_daily: late_arriving_count=10


[INFO] 09.CK.09 bronze.transaction_daily: posting_before_transaction_count=6


[INFO] 09.CK.10 bronze.transaction_daily: fx_tolerance_breach_count=37
[PASS] assessment1-profiling-task1: overall status=PASS


### Critical data elements

Nominated against the checks and joins actually exercised by this assessment's three tasks - a column earns the label because a specific check, reconciliation cut, or root-cause step depends on it, not because it appears in the schema.

| id | column                | why critical                                                    |
| -- | --------------------- | ----------------------------------------------------------------- |
| 01 | transaction_id        | business key - drives 09.CK.02 dedup and the source-to-Bronze join |
| 02 | transaction_amount    | feeds task 2 level 1 batch totals and the sign check 09.CK.06      |
| 03 | local_currency_amount | the balance Finance reported broken - task 2's reconciled measure  |
| 04 | exchange_rate         | ties amount to local_currency_amount - the 09.CK.10 tolerance check |
| 05 | currency_code         | dimensional reconciliation cut plus the 09.CK.05 validity check    |
| 06 | transaction_type      | debit/credit split for level 1 totals plus the 09.CK.05 validity check |
| 07 | transaction_date      | task 2 dimensional cut and task 3's business-date boundary          |
| 08 | posting_date          | 09.CK.09 ordering check plus accounting-date reconciliation         |
| 09 | branch_code           | task 2 level 2 dimensional reconciliation cut                       |
| 10 | product_code          | task 2 level 2 dimensional reconciliation cut                       |
| 11 | ingestion_file        | 09.CK.07 source-file distribution and task 3's root-cause trace     |
| 12 | source_extract_ts     | 09.CK.08 late-arriving check and task 3's UTC/SGT boundary evidence  |

`account_id` and `source_system` are excluded - neither is read by any Task 1-3 check, reconciliation cut, or root-cause step in this assessment's scope.


In [8]:
spark.stop()
